# Bank Marketing (FairDPGANs) 

Author: Ilse Harmers \
Last updated: February 17, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import warnings
warnings.filterwarnings("ignore", message = r"Using", category = FutureWarning)
warnings.filterwarnings("ignore", message = r"invalid", category = RuntimeWarning)

In [ ]:
# Importing train set.
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")

# Preprocessing the dataset for the Fair DP-GANs; the dataset Fair DP-GANs are expecting the first two columns in the original dataset to be the 
# sensitive and target attributes. 
cols = bank_train.columns.to_list()
cols.remove("age")
cols.remove("y")
bank_train = bank_train[["age", "y"] + cols]

print(bank_train.columns.to_list())
bank_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = bank_train["age"].min(), upper = bank_train["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # y
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # job
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # default
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-8000) + 1)), upper = np.log(102000 + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # housing
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # loan
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # contact
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # day
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # month
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(4900 + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = bank_train["campaign"].min(), upper = 60,
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_train["pdays"].min()) + 1)), upper = np.log(870 + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(280 + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # poutcome
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 4 * 10^4 rows, then delta = 10^(-5). 
delta = 10**np.floor(np.log10(1/bank_train.shape[0]))
print(delta)

# Defining beta1 in Adam optimizer.
beta1 = 0.9 

# Defining epsilon value.
epsi = 1

# Defining Fair DP-GAN: [dem, dis, dem-dis, clf_dem, clf_dis, clf_eqopp, clf_all].
fair_loss = "dem"

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")

    # Synthesizing the dataset with one of the Fair DP-GANs.
    synth = Synthesizer.create('fairdpgan', s = "age", s_unpriv = 25, s_priv = 25, y = "y", y_des = "yes", fair_loss = fair_loss, epochs = 10000,
                               epsilon = epsi, delta = delta, beta1 = beta1, plot_losses = True, batch_size = 512, dataset = "Bank",
                               file_path = f"synthetic-datasets_FairDP-GAN({fair_loss})_B=512/bank-marketing/epsi_{epsi}/run{r}/",
                               #sanity_check = True
                              )
    synth.fit(bank_train, transformer = tt, preprocessor_eps = 0.0)

    try:
        # Generating the first synthetic dataset.
        sample1 = synth.sample(bank_train.shape[0])
        # Encoding the target and sensitive attributes for the fairness analysis.
        y1_encoded = utils.one_hot_encode(sample1[["y"]], order = [["no", "yes"]])
        age1_encoded = sample1["age"].copy()
        age1_encoded.loc[age1_encoded <= 25] = 1
        age1_encoded.loc[age1_encoded > 25] = 0
        sample1_encoded = pd.concat([age1_encoded.reset_index(drop = True), y1_encoded.reset_index(drop = True)], axis = 1)
        dem1 = utils.demographic_parity(df = sample1_encoded, s = "age", y = "y")
        dis1 = utils.disparate_impact(df = sample1_encoded, s = "age", y = "y")
        print("Sample 1 [dem., dis.]: ",  [dem1, dis1])
        
        # Generating the second synthetic dataset.
        sample2 = synth.sample(bank_train.shape[0])
        # Encoding the target and sensitive attributes for the fairness analysis.
        y2_encoded = utils.one_hot_encode(sample2[["y"]], order = [["no", "yes"]])
        age2_encoded = sample2["age"].copy()
        age2_encoded.loc[age2_encoded <= 25] = 1
        age2_encoded.loc[age2_encoded > 25] = 0
        sample2_encoded = pd.concat([age2_encoded.reset_index(drop = True), y2_encoded.reset_index(drop = True)], axis = 1)
        dem2 = utils.demographic_parity(df = sample2_encoded, s = "age", y = "y")
        dis2 = utils.disparate_impact(df = sample2_encoded, s = "age", y = "y")
        print("Sample 2 [dem., dis.]: ",  [dem2, dis2])

        # Generating the third synthetic dataset.
        sample3 = synth.sample(bank_train.shape[0])
        # Encoding the target and sensitive attributes for the fairness analysis.
        y3_encoded = utils.one_hot_encode(sample3[["y"]], order = [["no", "yes"]])
        age3_encoded = sample3["age"].copy()
        age3_encoded.loc[age3_encoded <= 25] = 1
        age3_encoded.loc[age3_encoded > 25] = 0
        sample3_encoded = pd.concat([age3_encoded.reset_index(drop = True), y3_encoded.reset_index(drop = True)], axis = 1)
        dem3 = utils.demographic_parity(df = sample3_encoded, s = "age", y = "y")
        dis3 = utils.disparate_impact(df = sample3_encoded, s = "age", y = "y")
        print("Sample 3 [dem., dis.]: ",  [dem3, dis3])

        # Saving the synthetic datasets.
        sample1.to_csv(f"./synthetic-datasets_FairDP-GAN({fair_loss})_B=512/bank-marketing/epsi_{epsi}/run{r}/sample1.csv", index = False)
        sample2.to_csv(f"./synthetic-datasets_FairDP-GAN({fair_loss})_B=512/bank-marketing/epsi_{epsi}/run{r}/sample2.csv", index = False)
        sample3.to_csv(f"./synthetic-datasets_FairDP-GAN({fair_loss})_B=512/bank-marketing/epsi_{epsi}/run{r}/sample3.csv", index = False)
        
        r += 1
    except ZeroDivisionError:
        r += 0